# ❓ RAG System for Question Answering

Build the first full retrieve-then-generate pipeline: index three research papers, retrieve the chunks that match a
question, and let an LLM answer from those chunks only.

**Reference:** [RAG Systems Essentials (Analytics Vidhya)](https://courses.analyticsvidhya.com/courses/take/rag-systems-essentials/lessons/60148017-hands-on-deep-dive-into-rag-evaluation-metrics-generator-metrics-i)

## Learning Objectives
In this notebook, you will learn:
1. **Chunk research papers** - load PDFs and split them with overlap (contextual chunking as an optional variant)
2. **Index the chunks** - embed them into a persistent Chroma collection
3. **Retrieve** - similarity search and multi-query retrieval over the papers
4. **Generate grounded answers** - an LCEL RAG chain that answers only from the retrieved context

## Where this fits
`01_build_document_retriever_search_engine.ipynb` stopped at retrieval. This notebook adds the generation step, which
turns the search engine into a question-answering system.

## Prerequisites
- `01_build_document_retriever_search_engine.ipynb` (chunking, Chroma, retrievers) - background reading
- Dependencies installed with `uv sync` in this project's folder
- `OPENAI_API_KEY` in this project's `.env` (OpenAI embeddings and `gpt-4o-mini`)

---
## 📦 Step 1: Environment Setup

Load `OPENAI_API_KEY` from this project's `.env`, then create the chat model used for query rewriting and answer generation.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from this project's .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: gpt-4o-mini for query rewriting and answers
# ============================================================================
from aied_helpers import get_openai_llm

# temperature=0 is the factory default, as in the original lesson
llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---
## 📄 Part 1: Chunk the Research Papers

`data/` holds three papers: *Attention Is All You Need*, the RAG paper and the GPT-3 paper. Two chunking helpers follow:
`create_contextual_chunks` (the contextual-retrieval approach from `01_build_document_retriever_search_engine.ipynb`)
and `create_simple_chunks` (plain splitting with overlap). This notebook indexes the simple chunks; the contextual
variant is kept as an optional, commented-out cell because it sends a whole paper to the LLM for every chunk.

In [ ]:
# ============================================================================
# CHUNK CONTEXT (optional): An LLM chain that situates one chunk inside its paper
# ============================================================================
# create a chat prompt
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


def generate_chunk_context(document, chunk):

    chunk_process_prompt = """You are an AI assistant specializing in research paper analysis.
                            Your task is to provide brief, relevant context for a chunk of text
                            based on the following research paper.

                            Here is the research paper:
                            <paper>
                            {paper}
                            </paper>

                            Here is the chunk we want to situate within the whole document:
                            <chunk>
                            {chunk}
                            </chunk>

                            Provide a concise context (3-4 sentences max) for this chunk,
                            considering the following guidelines:

                            - Give a short succinct context to situate this chunk within the overall document
                            for the purposes of improving search retrieval of the chunk.
                            - Answer only with the succinct context and nothing else.
                            - Context should be mentioned like 'Focuses on ....'
                            do not mention 'this chunk or section focuses on...'

                            Context:
                        """

    prompt_template = ChatPromptTemplate.from_template(chunk_process_prompt)

    agentic_chunk_chain = (prompt_template
                                |
                            llm
                                |
                            StrOutputParser())

    context = agentic_chunk_chain.invoke({'paper': document, 'chunk': chunk})

    return context

In [ ]:
# ============================================================================
# CONTEXTUAL CHUNKS (optional): Load, split and contextualise one PDF file
# ============================================================================
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document


def create_contextual_chunks(file_path):

    print('Loading pages:', file_path)
    loader = PyMuPDFLoader(file_path)
    doc_pages = loader.load()

    print('Chunking pages:', file_path)
    splitter = RecursiveCharacterTextSplitter(chunk_size=3500,
                                              chunk_overlap=0)
    doc_chunks = splitter.split_documents(doc_pages)

    print('Generating contextual chunks:', file_path)
    original_doc = '\n'.join([doc.page_content for doc in doc_chunks])
    contextual_chunks = []
    for chunk in doc_chunks:
        context = generate_chunk_context(original_doc, chunk.page_content)
        contextual_chunks.append(Document(page_content=context+'\n'+chunk.page_content,
                                          metadata=chunk.metadata))
    print('Finished processing:', file_path)
    print()
    return contextual_chunks

In [ ]:
# ============================================================================
# SIMPLE CHUNKS: Load and split one PDF file
# ============================================================================
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

def create_simple_chunks(file_path, chunk_size=3500, chunk_overlap=0):

    print('Loading pages:', file_path)
    loader = PyMuPDFLoader(file_path)
    doc_pages = loader.load()

    print('Chunking pages:', file_path)
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size,
                                              chunk_overlap=chunk_overlap)
    doc_chunks = splitter.split_documents(doc_pages)
    print('Finished processing:', file_path)
    print()
    return doc_chunks

In [ ]:
from glob import glob

pdf_files = glob('../data/*.pdf')
pdf_files

In [ ]:
# ============================================================================
# CHUNK ALL PAPERS: 3,500-character chunks with 200 characters of overlap
# ============================================================================
paper_docs = []
for fp in pdf_files:
    paper_docs.extend(create_simple_chunks(file_path=fp,
                                           chunk_size=3500,
                                           chunk_overlap=200))

In [ ]:
len(paper_docs)

Optional: to index contextual chunks instead, uncomment the next cell. It makes one LLM call per chunk with the whole paper in the prompt; the 75-page GPT-3 paper alone makes this many times costlier than simple chunking.

In [ ]:
# paper_docs = []
# for fp in pdf_files:
#     paper_docs.extend(create_contextual_chunks(fp))

---
## 🗄️ Part 2: Index the Chunks in Chroma

Embed every chunk with `text-embedding-3-small` and store it in a persistent Chroma collection with cosine distance, then load it back from disk.

In [ ]:
# ============================================================================
# EMBEDDINGS: OpenAI text-embedding-3-small
# ============================================================================
from langchain_openai import OpenAIEmbeddings

# details here: https://openai.com/blog/new-embedding-models-and-api-updates
openai_embed_model = OpenAIEmbeddings(model='text-embedding-3-small')

In [ ]:
# ============================================================================
# VECTOR DB: Embed the paper chunks into a persistent Chroma collection
# ============================================================================
from pathlib import Path

from langchain_chroma import Chroma

# create vector DB of docs and embeddings - takes < 30 s
# from_documents on an existing folder would store every document a second time,
# so the vector DB is built only once - delete ./final_project to rebuild it
if Path("./final_project").exists():
    print("✅ ./final_project already exists - the next cell loads it (delete the folder to rebuild)")
else:
    chroma_db = Chroma.from_documents(documents=paper_docs,
                                      collection_name='final_project',
                                      embedding=openai_embed_model,
                                      # need to set the distance function to cosine else it uses euclidean by default
                                      # check https://docs.trychroma.com/guides#changing-the-distance-function
                                      collection_metadata={"hnsw:space": "cosine"},
                                      persist_directory="./final_project")
    print(f"✅ Indexed {len(paper_docs)} documents into ./final_project")

In [ ]:
# load from disk
chroma_db = Chroma(persist_directory="./final_project",
                   collection_name='final_project',
                   embedding_function=openai_embed_model)

---
## 🎯 Part 3: Retrieve Relevant Chunks

Start with plain similarity search (top 5), then wrap it in a `MultiQueryRetriever`, which asks the LLM for several rewrites of each question and unions their results.

In [ ]:
similarity_retriever = chroma_db.as_retriever(search_type="similarity",
                                              search_kwargs={"k": 5})

In [ ]:
# ============================================================================
# DISPLAY HELPER: Print metadata and the first 1,000 characters of each document
# ============================================================================
from IPython.display import display, Markdown

def display_docs(docs):
    for doc in docs:
        print('Metadata:', doc.metadata)
        print('Content Brief:')
        display(Markdown(doc.page_content[:1000]))
        print()

In [ ]:
query = "what is machine learning?"
top_docs = similarity_retriever.invoke(query)
display_docs(top_docs)

In [ ]:
# ============================================================================
# MULTI-QUERY RETRIEVER: The LLM writes several versions of each query
# ============================================================================
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
# Set logging for the queries
import logging

similarity_retriever = chroma_db.as_retriever(search_type="similarity",
                                              search_kwargs={"k": 5})

mq_retriever = MultiQueryRetriever.from_llm(
    retriever=similarity_retriever, llm=llm
)

logging.basicConfig()
# so we can see what queries are generated by the LLM
logging.getLogger("langchain_classic.retrievers.multi_query").setLevel(logging.INFO)

The `INFO` log lines under each query show the rewritten queries the LLM generated.

In [ ]:
query = "what is a cnn?"
top_docs = mq_retriever.invoke(query)
display_docs(top_docs)

In [ ]:
query = "what is nlp?"
top_docs = mq_retriever.invoke(query)
display_docs(top_docs)

---
## 🧠 Part 4: Generate Grounded Answers

The RAG prompt tells the model to answer only from the retrieved context and to say it doesn't know otherwise. The LCEL chain feeds the multi-query retriever's documents (joined into one string) and the question into that prompt, then into the LLM.

In [ ]:
# ============================================================================
# RAG PROMPT: Answer only from the retrieved context
# ============================================================================
from langchain_core.prompts import ChatPromptTemplate

rag_prompt = """You are an assistant who is an expert in question-answering tasks.
                Answer the following question using only the following pieces of retrieved context.
                If the answer is not in the context, do not make up answers, just say that you don't know.
                Keep the answer detailed and well formatted based on the information from the context.

                Question:
                {question}

                Context:
                {context}

                Answer:
            """

rag_prompt_template = ChatPromptTemplate.from_template(rag_prompt)

In [ ]:
# ============================================================================
# RAG CHAIN: retriever -> format_docs -> prompt -> LLM
# ============================================================================
from langchain_core.runnables import RunnablePassthrough
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

qa_rag_chain = (
    {
        "context": (mq_retriever
                      |
                    format_docs),
        "question": RunnablePassthrough()
    }
      |
    rag_prompt_template
      |
    llm
)

In [ ]:
query = "What is machine learning?"
result = qa_rag_chain.invoke(query)
display(Markdown(result.content))

In [ ]:
query = "What are the main components of a RAG model and how do they interact?"
result = qa_rag_chain.invoke(query)
display(Markdown(result.content))

Compare the two answers. The papers are about transformers, retrieval-augmented generation and GPT-3, not
machine learning in general, so the first answer can only be as broad as those papers allow. The second question is
answered directly by the RAG paper (`2005.11401v4.pdf`): retriever plus generator.

---
## 📝 Summary

In this notebook, we learned:

### 1. Chunk Research Papers
- `PyMuPDFLoader` plus `RecursiveCharacterTextSplitter` (with overlap) turns PDFs into retrievable chunks.
- Contextual chunking is an optional upgrade whose cost grows with document length.

### 2. Index the Chunks
- One `Chroma.from_documents` call embeds and persists the chunks; build once and reload from disk.

### 3. Retrieve
- Multi-query retrieval widens recall by searching with several LLM rewrites of the question.

### 4. Generate Grounded Answers
- An LCEL chain (`{context, question} | prompt | llm`) is the whole RAG pipeline; the prompt keeps answers grounded.

### Next Steps
- **`03_conversational_rag.ipynb`**: adds chat history so follow-up questions such as "tell me more about it" work.